# 2 — Features: calendário e meteorologia

Constrói a **base analítica** do estudo a partir do taxi-out do notebook `1`.
Tudo o que os modelos consomem é criado aqui.

1. Aplica a janela plausível de taxi-out e cria `log_taxi`;
2. Deriva as variáveis de calendário no fuso de Brasília;
3. Decodifica a meteorologia do METAR e casa cada voo com a observação vigente
   no pushback;
4. Infere a cabeceira em uso pelo vento e marca troca recente de cabeceira;
5. Monta as variáveis meteorológicas por voo.

As features são apenas **preditores disponíveis na cabine**.

Saída: `data/processed/cgh_taxi_features_2025.parquet`


In [24]:
import numpy as np
import pandas as pd

SRC_TAXI = "data/processed/cgh_taxi_out_2025.parquet"
SRC_METAR = "data/processed/cgh_metar_2025.parquet"
OUT = "data/processed/cgh_taxi_features_2025.parquet"

TZ = "America/Sao_Paulo"

# Janela plausivel de taxi-out (min). Serve como conferencia de sanidade: o
# notebook 1 ja resolve na origem os emparelhamentos falsos do join, entao
# poucos voos caem fora daqui.
TAXI_MIN, TAXI_MAX = 3.0, 90.0

DIAS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
DIAS_PT = ["Segunda", "Terca", "Quarta", "Quinta", "Sexta", "Sabado", "Domingo"]

# Granularidade do bloco de horario (min). Opcoes: 15, 30, 45, 60. Todas
# dividem 1440, entao alinham a meia-noite local. Trocar aqui e reexecutar.
BLOCO_MIN = 15

# --- Parametros da cabeceira (secao 2.5) --------------------------------------
# QFU magnetico das cabeceiras de CGH; o METAR reporta vento em graus VERDADEIROS,
# entao aplica-se a declinacao (21 W em 2025) para comparar com o vento.
QFU_MAG = {"17": 169.0, "35": 349.0}
DECL_W = 21.0
# Historese da troca: cauda > TROCA_KT troca ja; senao exige PERSIST observacoes
# consecutivas favorecendo a outra pista. VRB/calmaria herdam a cabeceira atual.
TROCA_KT = 2.0
PERSIST = 2
# Janela (min) para a flag "houve troca de cabeceira antes do pushback".
# Parametro exploratorio: trocar aqui e reexecutar para testar outras janelas.
JANELA_TROCA_MIN = 60


## 2.1 Janela plausível de taxi-out

O corte de 3 a 90 min elimina possiveis erros residuais de casamento.
`log_taxi` é a variável dependente de toda a modelagem: o taxi-out é uma
duração positiva e assimétrica, e o logaritmo normaliza os resíduos e converte
os coeficientes em variação percentual.


In [25]:
df = pd.read_parquet(SRC_TAXI)
df["taxi_out_min"] = df["taxi_out_min"].astype("float64")
n0 = len(df)

excl = pd.DataFrame({
    "criterio": [f"taxi_out < {TAXI_MIN:.0f} min", f"taxi_out > {TAXI_MAX:.0f} min"],
    "n": [int((df["taxi_out_min"] < TAXI_MIN).sum()),
          int((df["taxi_out_min"] > TAXI_MAX).sum())],
})
excl["pct"] = (excl["n"] / n0 * 100).round(3)
display(excl)

df = df[(df["taxi_out_min"] >= TAXI_MIN) & (df["taxi_out_min"] <= TAXI_MAX)].copy()
df["log_taxi"] = np.log(df["taxi_out_min"])

print(f"registros brutos:  {n0}")
print(f"apos a janela:     {len(df)} ({len(df) / n0 * 100:.1f}%)")
print(f"mediana:           {df['taxi_out_min'].median():.1f} min")


,criterio,n,pct
0,taxi_out < 3 min,35,0.047
1,taxi_out > 90 min,1,0.001


registros brutos:  75055
apos a janela:     75019 (100.0%)
mediana:           13.9 min


## 2.2 Variáveis de calendário

Todas derivadas do pushback (`partida_real`) convertido para o **horário de
Brasília** — é o relógio que governa a operação do aeroporto e a grade das
companhias.

- `mes`, `dia_semana` (categórico ordenado seg→dom);
- `bloco`: faixa do dia com passo `BLOCO_MIN` (definido no topo). O período
  operacional é **06h–23h**; os voos antes das 06h (madrugada, esparsos) são
  agrupados em `<06:00`. 
- `cia_aerea`: as três primeiras letras do callsign (prefixo ICAO da empresa);
- `data`: data local em texto, usada como **identificador do dia** — o nível 2
  do modelo multinível.


In [26]:
local = df["partida_real"].dt.tz_convert(TZ)

df["data"] = local.dt.date.astype(str)
df["mes"] = local.dt.month

df["dia_semana"] = (
    pd.Categorical(local.dt.day_name(), categories=DIAS, ordered=True)
    .rename_categories(DIAS_PT)
)


def faixas(step_min):
    """Blocos de `step_min` min no periodo 06h-23h; madrugada agrupada."""
    b = local.dt.floor(f"{step_min}min").dt.strftime("%H:%M")
    b = b.mask(local.dt.hour < 6, "<06:00")
    meio = sorted(x for x in b.unique() if x != "<06:00")
    ordem = ["<06:00"] + meio
    ordem = [x for x in ordem if x in set(b)]
    return pd.Categorical(b, categories=ordem, ordered=True)


df["bloco"] = faixas(BLOCO_MIN)

df["cia_aerea"] = pd.Categorical(df.index.str[0:3])

print(f"bloco de {BLOCO_MIN} min:", df["bloco"].nunique(), "niveis")
print("3 blocos mais ralos:",
      df["bloco"].value_counts().sort_values().head(3).to_dict())
print("cia_aerea:", df["cia_aerea"].value_counts().to_dict())
print("dias distintos:", df["data"].nunique())


bloco de 15 min: 73 niveis
3 blocos mais ralos: {'23:45': 2, '23:00': 5, '23:15': 10}
cia_aerea: {'TAM': 34604, 'GLO': 31676, 'AZU': 8682, 'PTB': 57}
dias distintos: 365


## 2.3 Feriados

`feriado` marca o feriado, a **véspera** e o **pós-feriado**, porque o efeito
sobre a demanda aérea se espalha pelos dias vizinhos: a viagem costuma
acontecer na véspera e o retorno no dia seguinte.

Feriados que caem em sábado ou domingo **não** entram, nem geram véspera ou
pós-feriado — o fim de semana já tem um padrão de tráfego próprio, e misturá-los
confundiria os dois efeitos. Pontos facultativos de grande adesão (Carnaval,
Corpus Christi) são tratados como feriado.


In [27]:
FERIADOS_2025 = {
    "2025-01-01": "Ano Novo",
    "2025-01-25": "Aniversario de Sao Paulo",     # municipal (cidade de SP)
    "2025-03-03": "Carnaval",                     # ponto facultativo
    "2025-03-04": "Carnaval",                     # ponto facultativo
    "2025-04-18": "Sexta-feira Santa",
    "2025-04-21": "Tiradentes",
    "2025-05-01": "Dia do Trabalho",
    "2025-06-19": "Corpus Christi",               # ponto facultativo
    "2025-07-09": "Revolucao Constitucionalista", # estadual (SP)
    "2025-09-07": "Independencia",
    "2025-10-12": "N. Sra. Aparecida",
    "2025-11-02": "Finados",
    "2025-11-15": "Proclamacao da Republica",
    "2025-11-20": "Consciencia Negra",
    "2025-12-25": "Natal",
}

feriados = pd.to_datetime(list(FERIADOS_2025)).normalize()
feriados_uteis = feriados[feriados.dayofweek < 5]   # exclui os que caem em sab/dom

_data = local.dt.tz_localize(None).dt.normalize()
_fer = _data.isin(feriados_uteis)
_vesp = (_data + pd.Timedelta(days=1)).isin(feriados_uteis)
_pos = (_data - pd.Timedelta(days=1)).isin(feriados_uteis)
df["feriado"] = (_fer | _vesp | _pos).astype("int8")

print(f"feriados no ano: {len(feriados)} | em dia util: {len(feriados_uteis)}")
print("voos por categoria:", df["feriado"].value_counts().to_dict())


feriados no ano: 15 | em dia util: 10
voos por categoria: {0: 69926, 1: 5093}


## 2.4 Variáveis meteorológicas a partir do METAR

O METAR decodificado (notebook `0_3`) traz uma observação a cada hora, em
**UTC** (ou instantanea se houver algum evento significativo). Desta seção saem as variáveis meterorologicas:

- **`teto_ft`** — o teto é a base da camada mais baixa com cobertura `BKN`(Broken-Nublado) ou
  `OVC`(Overcast - Encoberto). Nuvem `FEW`/`SCT` (Few/Scattered - Poucas/Parcialmente nublado) não constitui teto, então as camadas são  varridas em ordem e a menor altura qualificada vence;
- **`chuva`** — categórica ordinal `sem / leve / moderada / forte`. Trovoada é lidada à parte:
  `TS` em qualquer posição marca `trovoada`.
- **`vento_rajada`** — presença de rajada declarada (Gust).

**Direção do vento não é interpolada.** Ela é circular: interpolar entre 350° e
010° daria 180° — o vento contrário. Além disso, os ~7% de observações com
direção ausente **não** são falta de dado, e sim `VRB` (vento variável, quase
sempre fraco), que a seção 2.5 trata como estado próprio. Já `wind_speed` e
`visibility`, que são magnitudes, podem ser interpolados para cobrir faltas
isoladas na série semi-horária.


In [28]:
m = pd.read_parquet(SRC_METAR)
m["dt"] = m["date_time"].dt.tz_localize("UTC")   # METAR e sempre UTC

# Interpola so magnitudes; a direcao (circular) fica intacta -- ver secao 2.5.
for col in ["wind_speed", "visibility"]:
    m[col] = m[col].interpolate()

CAMADAS = [("low_cloud_type", "low_cloud_level"),
           ("medium_cloud_type", "medium_cloud_level"),
           ("high_cloud_type", "high_cloud_level"),
           ("highest_cloud_type", "highest_cloud_level")]


def ceiling_ft(row):
    """Teto = base da camada mais baixa com BKN ou OVC (FEW/SCT nao contam)."""
    niveis = [row[lvl] for tipo, lvl in CAMADAS
              if str(row[tipo]) in ("BKN", "OVC") and pd.notna(row[lvl])]
    return min(niveis) if niveis else np.nan


m["teto_ft"] = m.apply(ceiling_ft, axis=1)

wx = m[["current_wx1", "current_wx2", "current_wx3"]]

# Chuva: variavel categorica ordinal, exaustiva e exclusiva. A intensidade e a
# MAIS forte presente na observacao (aplica-se fraca -> forte, a ultima vence).
INTENSIDADE = [
    ("leve",     ["-DZ", "DZ", "+DZ", "-RA", "-TSRA"]),
    ("moderada", ["RA", "SHRA", "TSRA"]),
    ("forte",    ["+RA", "+TSRA"]),
]
chuva = pd.Series("sem", index=m.index, dtype=object)
for nome, codigos in INTENSIDADE:
    chuva = chuva.mask(wx.isin(codigos).any(axis=1), nome)
m["chuva"] = pd.Categorical(chuva, categories=["sem", "leve", "moderada", "forte"],
                            ordered=True)

# Trovoada e rajada sao genuinamente binarias (ha / nao ha), nao ordinais.
m["trovoada"] = (wx.astype("string")
                   .apply(lambda c: c.str.contains("TS", na=False))
                   .any(axis=1).astype(int))
m["vento_rajada"] = m["wind_gust"].notna().astype(int)

print("chuva (obs):", m["chuva"].value_counts().reindex(
      ["sem", "leve", "moderada", "forte"]).to_dict())
print(f"trovoada: {int(m['trovoada'].sum())} | rajada: {int(m['vento_rajada'].sum())}")
print(f"{len(m)} observacoes METAR | teto BKN/OVC presente em "
      f"{m['teto_ft'].notna().mean()*100:.0f}%")


chuva (obs): {'sem': 10404, 'leve': 803, 'moderada': 168, 'forte': 72}
trovoada: 264 | rajada: 238
11447 observacoes METAR | teto BKN/OVC presente em 53%


## 2.5 Cabeceira em uso e troca de cabeceira

Congonhas tem uma pista, operada em dois sentidos — cabeceiras **17** e **35**.
O sentido em uso não é observado; é **inferido do vento**.

Observações:

- **Vento verdadeiro × pista magnética.** O METAR reporta o vento em graus
  verdadeiros; a numeração da pista é magnética. Em CGH a declinação é ~21°W, fato este ja calculado.
- A pista não troca porque o vento girou 5° por meia hora —
  trocar é uma manobra cara. A regra: troca imediata se o vento de cauda na
  pista atual passa de `TROCA_KT`; se for marginal, exige `PERSIST` (N) observações
  seguidas favorecendo a outra; `VRB`/calmaria **herdam** a cabeceira vigente.
  Sem isso, ~29% das "trocas" durariam uma única observação (ruído de vento
  perto do limiar).

`troca_cabeceira` marca os voos cujo pushback ocorreu até `JANELA_TROCA_MIN`
depois de uma troca — a fila de reconfiguração ainda se desfazendo. É
informação disponível na cabine.


In [29]:
QFU = {r: (mag - DECL_W) % 360 for r, mag in QFU_MAG.items()}  # graus verdadeiros
OUTRA = {"17": "35", "35": "17"}

ms = m.sort_values("dt").reset_index(drop=True)
dir_, spd = ms["wind_direction"], ms["wind_speed"]
sem_info = dir_.isna() | (spd == 0)   # VRB ou calmaria: nao definem cabeceira


def cauda(cab, i):
    """Vento de cauda (kt) na cabeceira cab na observacao i; negativo = de proa."""
    return -spd[i] * np.cos(np.radians(dir_[i] - QFU[cab]))


# Cabeceira inicial: primeira obs com vento -> a que o vento favorece.
i0 = int(np.argmax(~sem_info.to_numpy()))
atual = "17" if cauda("17", i0) <= cauda("35", i0) else "35"

cab_uso, pend = [], 0
for i in range(len(ms)):
    if not sem_info[i]:
        tw = cauda(atual, i)
        if tw > TROCA_KT:            # cauda forte: troca agora
            atual = OUTRA[atual]; pend = 0
        elif tw > 0:                 # outra favorecida, mas fraco: confirma
            pend += 1
            if pend >= PERSIST:
                atual = OUTRA[atual]; pend = 0
        else:                        # atual ainda com proa
            pend = 0
    cab_uso.append(atual)

ms["cabeceira"] = pd.Categorical(cab_uso, categories=["17", "35"])

cab_str = ms["cabeceira"].astype(str)
trocou = cab_str.ne(cab_str.shift())
trocou.iloc[0] = False               # a primeira observacao nao e uma troca
trocas_dt = ms.loc[trocou, "dt"]

print("repartição da cabeceira:",
      ms["cabeceira"].value_counts(normalize=True).mul(100).round(1).to_dict())
print(f"trocas no ano: {len(trocas_dt)} (~{len(trocas_dt)/365:.2f}/dia)")
dur_h = ms.groupby(trocou.cumsum())["dt"].agg(
    lambda s: (s.max() - s.min()).total_seconds() / 3600)
print(f"duracao mediana de um bloco de mesma cabeceira: {dur_h.median():.1f} h")


repartição da cabeceira: {'17': 74.3, '35': 25.7}
trocas no ano: 491 (~1.35/dia)
duracao mediana de um bloco de mesma cabeceira: 6.0 h


## 2.6 Casamento METAR → voo, no pushback

Cada voo recebe a **observação válida mais recente antes do seu pushback**
(`merge_asof` para trás), que é a informação de que o piloto e o controle
dispunham no momento em que o taxi começou. A tolerância de 90 min cobre
eventuais falhas na série sem permitir que um voo herde um tempo velho demais.
A cabeceira em uso vem no mesmo casamento; a flag `troca_cabeceira` é calculada
comparando o pushback de cada voo com os instantes de troca.


In [30]:
METEO = ["visibility", "teto_ft", "wind_speed", "wind_direction","cabeceira",
         "chuva", "trovoada", "vento_rajada"]

df = pd.merge_asof(
    df.sort_values("partida_real"),
    ms[["dt"] + METEO].sort_values("dt"),
    left_on="partida_real", right_on="dt",
    direction="backward", tolerance=pd.Timedelta("90min"),
)

casou = df["dt"].notna()
print(f"casamento: {casou.mean()*100:.1f}%  "
      f"({(~casou).sum()} voos sem METAR em 90 min, descartados)")
df = df[casou].copy()
df = df.rename(columns={"cabeceira": "cabeceira_em_uso"})


def epoch_s(t):
    """Segundos desde a epoca, robusto a timestamps tz-aware em us."""
    return pd.to_datetime(t, utc=True).to_numpy().astype("datetime64[s]").astype("int64")


# Houve troca nos JANELA_TROCA_MIN minutos antes do pushback?
troca_s = np.sort(epoch_s(trocas_dt))
push_s = epoch_s(df["partida_real"])
janela_s = JANELA_TROCA_MIN * 60
il = np.searchsorted(troca_s, push_s - janela_s, side="right")
ir = np.searchsorted(troca_s, push_s, side="right")
df["troca_cabeceira"] = ((ir - il) > 0).astype(int)

print(f"cabeceira no pushback:",
      df["cabeceira_em_uso"].value_counts(normalize=True).mul(100).round(1).to_dict())
print(f"voos com troca nos {JANELA_TROCA_MIN} min antes do pushback: "
      f"{int(df['troca_cabeceira'].sum())} ({df['troca_cabeceira'].mean()*100:.1f}%)")


casamento: 100.0%  (6 voos sem METAR em 90 min, descartados)


C:\Users\cmtem\AppData\Local\Temp\ipykernel_132\1062381043.py:20: UserWarning: no explicit representation of timezones available for np.datetime64
  return pd.to_datetime(t, utc=True).to_numpy().astype("datetime64[s]").astype("int64")
C:\Users\cmtem\AppData\Local\Temp\ipykernel_132\1062381043.py:20: UserWarning: no explicit representation of timezones available for np.datetime64
  return pd.to_datetime(t, utc=True).to_numpy().astype("datetime64[s]").astype("int64")


cabeceira no pushback: {'17': 68.8, '35': 31.2}
voos com troca nos 60 min antes do pushback: 4020 (5.4%)


## 2.7 Variáveis meteorológicas de nível 1 (por voo)

Visibilidade, teto e vento são contínuos, mas entram em **faixas**, e não como
valor bruto, porque o efeito operacional não é linear: o que muda o procedimento
é cruzar um limite, não perder cem metros de visibilidade ou ganhar um nó de
vento. Cada um vira uma **categórica ordinal**, no mesmo estilo da chuva — uma
coluna por variável, com o **bom tempo como referência**.

**Os cortes são operacionais, definidos a priori** (não escolhidos para maximizar
ajuste):

- **Visibilidade**: `<3000 / 3000–5000 / ≥5000 m` (ref). 5000 m é o piso do voo
  visual; abaixo de 3000 m entram mínimos mais restritivos.
- **Teto**: `<500 / 500–1500 / >1500 ft` (ref). 500 ft e 1500 ft marcam a
  transição para procedimentos de precisão e o regime de voo por instrumento.
- **Vento**: `<15 (ref) / 15–21 / ≥21 kt`. 15 kt é a faixa em que o vento começa
  a pesar na operação; coincide com o joelho da relação empírica (ver análise).


In [31]:
# Visibilidade, teto e vento como categoricas ordinais (bom tempo = referencia,
# a primeira categoria). Teto ausente (sem BKN/OVC) = ceu alto = bom tempo.
df["vis_cat"] = pd.Categorical(
    np.select([df["visibility"] < 3000, df["visibility"] < 5000],
              ["lt3000", "3000_5000"], default="ge5000"),
    categories=["ge5000", "3000_5000", "lt3000"], ordered=True)

df["teto_cat"] = pd.Categorical(
    np.select([df["teto_ft"] < 500, df["teto_ft"] < 1500],
              ["lt500", "500_1500"], default="ge1500"),
    categories=["ge1500", "500_1500", "lt500"], ordered=True)

df["vento_cat"] = pd.Categorical(
    np.select([df["wind_speed"] >= 21, df["wind_speed"] >= 15],
              ["ge21", "15_21"], default="lt15"),
    categories=["lt15", "15_21", "ge21"], ordered=True)

for col in ["vis_cat", "teto_cat", "vento_cat", "chuva"]:
    vc = df.groupby(col, observed=True).agg(
        voos=(col, "size"), dias=("data", "nunique"))
    print(f"\n{col}:")
    print(vc.to_string())
print("\nFaixas com poucos voos/dias rendem coeficientes imprecisos;",
      "reportar n ao interpretar (ex.: chuva forte, vento >=21 kt).")



vis_cat:
            voos  dias
vis_cat               
ge5000     73886   365
3000_5000   1022    73
lt3000       105    22

teto_cat:
           voos  dias
teto_cat             
ge1500    58270   359
500_1500  16139   229
lt500       604    33

vento_cat:
            voos  dias
vento_cat             
lt15       72963   365
15_21       1847    76
ge21         203     6

chuva:
           voos  dias
chuva                
sem       71319   365
leve       3235    99
moderada    385    42
forte        74    19

Faixas com poucos voos/dias rendem coeficientes imprecisos; reportar n ao interpretar (ex.: chuva forte, vento >=21 kt).


## 2.8 Gravação da base analítica

Duas limpezas antes de gravar:

1. **Remoção de NaN nos preditores do modelo** (`MODELO_COLS`) — feita aqui, e
   não na modelagem: no `statsmodels`, um `NaN` faz o `patsy` descartar a linha
   sem que o vetor de grupos acompanhe, desalinhando o nível 2 do multinível.
2. **Seleção explícita das colunas** (`GRAVAR`) — grava só os preditores
   disponíveis na cabine, os alvos e a âncora temporal.



In [32]:
# Preditores do modelo (disponiveis na cabine). O agrupamento por dia (nivel 2)
# nao entra aqui: e re-derivado de partida_real na modelagem.
MODELO_COLS = ["log_taxi", "mes", "dia_semana", "bloco", "feriado", "cia_aerea",
               "cabeceira_em_uso", "troca_cabeceira",
               "chuva", "vis_cat", "teto_cat", "vento_cat",
               "trovoada", "vento_rajada"]

# Grava alvos + ancora temporal + preditores de cabine.
GRAVAR = (["taxi_out_min", "log_taxi", "partida_real",
           "mes", "dia_semana", "bloco", "cia_aerea", "feriado",
           "cabeceira_em_uso", "troca_cabeceira",
           "chuva", "vis_cat", "teto_cat", "vento_cat", "trovoada", "vento_rajada"])

n_antes = len(df)
n_dias = df["data"].nunique()
df = df.dropna(subset=MODELO_COLS).reset_index(drop=True)
print(f"removidos por NaN em preditor: {n_antes - len(df)}")

descartadas = [c for c in df.columns if c not in GRAVAR]
df = df[GRAVAR]
df.to_parquet(OUT, index=False)

print(f"colunas descartadas: {descartadas}")
print(f"\nsalvo: {OUT}")
print(f"{len(df)} voos | {df.shape[1]} colunas | {n_dias} dias")
print(f"mediana do taxi-out: {df['taxi_out_min'].median():.1f} min")


removidos por NaN em preditor: 0
colunas descartadas: ['callsign', 'day', 'decolagem', 'partida_prevista', 'taxi_out', 'data', 'dt', 'visibility', 'teto_ft', 'wind_speed', 'wind_direction']

salvo: data/processed/cgh_taxi_features_2025.parquet
75013 voos | 16 colunas | 365 dias
mediana do taxi-out: 13.9 min


In [33]:
df.head()

,taxi_out_min,log_taxi,partida_real,mes,dia_semana,bloco,cia_aerea,feriado,cabeceira_em_uso,troca_cabeceira,chuva,vis_cat,teto_cat,vento_cat,trovoada,vento_rajada
0,11.750000,2.463853,2025-01-01 09:00:00+00:00,1,Quarta,06:00,TAM,1,17,0,sem,ge5000,ge1500,lt15,0.0,0.0
1,13.316667,2.589016,2025-01-01 09:13:00+00:00,1,Quarta,06:00,TAM,1,17,0,sem,ge5000,ge1500,lt15,0.0,0.0
2,15.216667,2.722391,2025-01-01 09:23:00+00:00,1,Quarta,06:15,TAM,1,17,0,sem,ge5000,ge1500,lt15,0.0,0.0
3,13.816667,2.625876,2025-01-01 09:31:00+00:00,1,Quarta,06:30,TAM,1,17,0,sem,ge5000,ge1500,lt15,0.0,0.0
4,13.583333,2.608844,2025-01-01 09:47:00+00:00,1,Quarta,06:45,TAM,1,17,0,sem,ge5000,ge1500,lt15,0.0,0.0


In [34]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 75013 entries, 0 to 75012
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   taxi_out_min      75013 non-null  float64            
 1   log_taxi          75013 non-null  float64            
 2   partida_real      75013 non-null  datetime64[us, UTC]
 3   mes               75013 non-null  int32              
 4   dia_semana        75013 non-null  category           
 5   bloco             75013 non-null  category           
 6   cia_aerea         75013 non-null  category           
 7   feriado           75013 non-null  int8               
 8   cabeceira_em_uso  75013 non-null  category           
 9   troca_cabeceira   75013 non-null  int64              
 10  chuva             75013 non-null  category           
 11  vis_cat           75013 non-null  category           
 12  teto_cat          75013 non-null  category           
 13  vento_cat   